# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [33]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [34]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_tiempo
dim_tiempo = spotify_raw[['Release Date']].drop_duplicates().reset_index(drop=True)
dim_tiempo['id_release'] = dim_tiempo.index + 1
print(dim_tiempo.head())

  Release Date  id_release
0    4/26/2024           1
1     5/4/2024           2
2    3/19/2024           3
3    1/12/2023           4
4    5/31/2024           5


In [35]:
# 2. Crear dim_pista
dim_pista = spotify_raw[['Track', 'Artist', 'Explicit Track']].drop_duplicates().reset_index(drop=True)
dim_pista['id_track'] = dim_pista.index + 1 
print(dim_pista.head())

                        Track          Artist  Explicit Track  id_track
0         MILLION DOLLAR BABY   Tommy Richman               0         1
1                 Not Like Us  Kendrick Lamar               1         2
2  i like the way you kiss me         Artemas               0         3
3                     Flowers     Miley Cyrus               0         4
4                     Houdini          Eminem               1         5


In [38]:
# 3. Crear fact_streams

# Cruzar IDs
df_temp = spotify_raw.merge(dim_tiempo, on='Release Date', how='left')
df_temp = df_temp.merge(dim_pista, on=['Track', 'Artist', 'Explicit Track'], how='left')
df_temp.head(5)

,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track,id_release,id_track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0,1,1
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1,2,2
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0,3,3
3,Flowers,Flowers - Single,Miley Cyrus,1/12/2023,USSM12209777,4,444.9,"2,031,280,633","269,802","136,569,078",...,"24,684,248",210.0,"190,260,277","203,384",NaN,"11,822,942",NaN,0,4,4
4,Houdini,Houdini,Eminem,5/31/2024,USUG12403398,5,423.3,"107,034,922","7,223","151,469,874",...,"17,660,624",105.0,"4,493,884","7,006","207,179","457,017",NaN,1,5,5


En este paso cometi un error, al hacer el merge de dimensión pista solo puse on='Track', por lo que si habia mas de un registro para una cancion, por ejemplo colaboraciones donde la cancion se guarda individualmente con los nombres de cada artista,  se volvian a emparejar en el df.

In [40]:
fact_streams = df_temp[["id_release", "id_track", "Spotify Streams", "YouTube Views", "Track Score"]] 
print("Data warehouse cargado")
fact_streams.head()

Data warehouse cargado


,id_release,id_track,Spotify Streams,YouTube Views,Track Score
0,1,1,"390,470,936","84,274,754",725.4
1,2,2,"323,703,884","116,347,040",545.9
2,3,3,"601,309,283","122,599,116",538.4
3,4,4,"2,031,280,633","1,096,100,899",444.9
4,5,5,"107,034,922","77,373,957",423.3


In [42]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
cubo_olap = fact_streams.merge(dim_tiempo, on='id_release').merge(dim_pista, on='id_track')
cubo_olap

,id_release,id_track,Spotify Streams,YouTube Views,Track Score,Release Date,Track,Artist,Explicit Track
0,1,1,"390,470,936","84,274,754",725.4,4/26/2024,MILLION DOLLAR BABY,Tommy Richman,0
1,2,2,"323,703,884","116,347,040",545.9,5/4/2024,Not Like Us,Kendrick Lamar,1
2,3,3,"601,309,283","122,599,116",538.4,3/19/2024,i like the way you kiss me,Artemas,0
3,4,4,"2,031,280,633","1,096,100,899",444.9,1/12/2023,Flowers,Miley Cyrus,0
4,5,5,"107,034,922","77,373,957",423.3,5/31/2024,Houdini,Eminem,1
...,...,...,...,...,...,...,...,...,...
4595,1497,4532,"305,049,963","149,247,747",19.4,9/5/2017,For the Last Time,$uicideboy$,1
4596,1030,4533,"52,282,360","943,920,245",19.4,7/27/2018,Dil Meri Na Sune,Atif Aslam,0
4597,746,4534,"189,972,685","201,027,333",19.4,2/28/2020,Grace (feat. 42 Dugg),Lil Baby,1
4598,1561,4535,"145,467,020","1,118,595,159",19.4,11/8/2016,Nashe Si Chadh Gayi,Arijit Singh,0


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [ ]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: [Escribe aquí si es Roll-up, Drill-down, Slice o Dice]
respuesta_1 = # Tu código...

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: [Escribe aquí...]
respuesta_2 = # Tu código...

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: [Escribe aquí...]
respuesta_3 = # Tu código...

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: [Escribe aquí...]
respuesta_4 = # Tu código...

# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: [Escribe aquí...]
respuesta_5 = # Tu código...

In [ ]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: [Escribe aquí...]
respuesta_6 = # Tu código...

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: [Escribe aquí...]
respuesta_7 = # Tu código...

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: [Escribe aquí...]
respuesta_8 = # Tu código...

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: [Escribe aquí...]
respuesta_9 = # Tu código...

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: [Escribe aquí...]
respuesta_10 = # Tu código...